# Lecture 5 — Expectations and Transition Dynamics
## 第五讲 —— 预期与过渡动态

**Computational Methods for Heterogeneous-Agent Macro**
**异质性主体宏观的计算方法**

Jeffrey Sun

## 1. Setup / 准备

We load `HouseholdStages` together with plotting and `Accessors` (for
the `@set` macro used in comparative statics).

加载 `HouseholdStages`、绘图工具，以及 `Accessors`（用 `@set` 做比较静态）。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages, Plots, Printf, Accessors

  Activating project at `~/projects/research/CV is all you need`


## 2. The Aiyagari model in `HouseholdStages`
## 2. 用 `HouseholdStages` 复现 Aiyagari

Same three-stage household problem as L04, plus a Cobb-Douglas firm
and equilibrium $\bar K = \bar K^{\mathrm{supplied}}$. TFP `A` enters
through `aiyagari_prices`; the household chain's env carries only
prices `(r, w)`.

与 L04 相同的三阶段家庭问题，加上 Cobb-Douglas 企业，
均衡条件 $\bar K = \bar K^{\mathrm{supplied}}$。
全要素生产率 `A` 通过 `aiyagari_prices` 进入；
家庭链的 env 只携带价格 `(r, w)`。

### 2.1 Parameters / 参数

In [45]:
@kwdef struct AiyagariParams
    β ::Float64 = 0.96
    σ ::Float64 = 1.5
    α ::Float64 = 0.36
    δ ::Float64 = 0.08
    L ::Float64 = 1.0
    A ::Float64 = 1.0
    y_grid ::Vector{Float64} = [0.6, 1.0, 1.4]
    P_y    ::Matrix{Float64} = [0.7 0.2 0.1;
                                 0.2 0.6 0.2;
                                 0.1 0.2 0.7]
    N_w   ::Int     = 400
    w_min ::Float64 = 0.0
    w_max ::Float64 = 100.0
end

AiyagariParams

### 2.2 Layout and household chain / 状态空间与家庭链

Three stages, in time order:

1. **Markov income shock** — `MarkovStage`.
2. **Receive income** — `WealthChangeStage`, with the
   period-budget identity $w' = (1+r)\,w + w\cdot y$.
3. **Consumption-savings choice** — `ConsumptionSavingsStage` with
   CRRA utility.

Compose them with `∘` (time order: leftmost runs first), then
`define_moments!` attaches the aggregate-wealth integral that the
outer tatonnement reads.

三个阶段，按时间顺序排列：

1. **Markov 收入冲击** —— `MarkovStage`。
2. **领取收入** —— `WealthChangeStage`，使用预算恒等式
   $w' = (1+r)\,w + w\cdot y$。
3. **消费—储蓄选择** —— 带 CRRA 效用的 `ConsumptionSavingsStage`。

用 `∘` 把它们按时间顺序串起来（最左边的最先执行），
再用 `define_moments!` 把外层 tâtonnement 要读的总财富积分挂上去。

In [46]:
aiyagari_layout(p) = StateLayout(
    StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length = p.N_w, spacing = :log)),
    StateAxis(:income, p.y_grid),
)

_u_crra(c, ::Val{1}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ) = c < 0 ? -Inf : _u_crra(c, valσ)

function aiyagari_household(p::AiyagariParams)
    layout = aiyagari_layout(p)

    # Stage 1 — Markov income shock
    # 第一阶段 —— Markov 收入冲击
    shock = MarkovStage(layout; axis = :income, transition = p.P_y)

    # Stage 2 — Receive income
    # 第二阶段 —— 领取收入
    receipt = WealthChangeStage(layout;
        wealth_post = (cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.income,
    )

    # Stage 3 — Consumption-savings
    # 第三阶段 —— 消费—储蓄
    savings = ConsumptionSavingsStage(layout;
        β               = p.β,
        utility         = (cell, c; env) -> u_crra(c, Val(p.σ)),
        monotone_search = :divide_conquer,
    )

    # Compose in time order — leftmost runs first
    # 按时间顺序复合 —— 最左边的先运行
    hh = shock ∘ receipt ∘ savings

    # Attach the K_supplied moment (aggregate wealth integral at end of chain)
    # 挂上 K_supplied 矩（链末端的总财富积分）
    return define_moments!(hh; K_supplied = at_end(integrand = :wealth, reduce = sum))
end

# Cobb-Douglas factor prices; A is read from p, so the env stays (r, w).
# Cobb-Douglas 要素价格；A 从 p 读，env 保持为 (r, w)。
function aiyagari_prices(K, p::AiyagariParams)
    (; α, δ, L, A) = p
    return (;
        r = A * α * (K / L)^(α - 1) - δ,
        w = A * (1 - α) * (K / L)^α,
    )
end

aiyagari_prices (generic function with 1 method)

### 2.3 What does `hh` look like? / `hh` 长什么样？

A bundled `ChainStage` carries one Spec (pure configuration: layouts,
transition matrix, closures, attached moments) and one Buffer
(per-call state: kernels, scratch arrays, warm-start V/Λ, kernel
cache). Users never touch the Buffer directly — they just pass `hh`
into the package's helpers.

一个 `ChainStage` 包两样东西：一个 Spec（纯配置：布局、转移矩阵、
闭包、挂上去的矩）和一个 Buffer（每次调用的状态：核、临时数组、
热启动 V/Λ、核缓存）。用户不直接碰 Buffer——只把 `hh` 传给包里的
辅助函数。

In [47]:
hh = aiyagari_household(AiyagariParams())
dump(hh; maxdepth = 1)

UndefVarError: UndefVarError: `define_moments!` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### 2.4 Single-env probe / 单个 env 试解一下

The user-facing surface for "solve at a single env" is
`solve_steady_state_given_env!(hh, env)`. It runs the V backward iteration and
the Λ forward iteration, returns copies of `V`, `Λ`, and the
defined `moments` — and incidentally warm-starts the chain's buffer
for the next call.

Build the env with `make_env(hh; ...)`. The helper validates field
names against the chain's schema and raises a clear error if any
required key is missing.

单个 env 的用户接口是 `solve_steady_state_given_env!(hh, env)`：跑完 V 的后向
迭代与 Λ 的前向迭代，返回 `V`、`Λ` 的拷贝以及之前定义的 `moments`，
并顺带为链的 buffer 设好热启动。

用 `make_env(hh; ...)` 构造 env：它会拿链的 schema 校验字段名，
缺哪个就报哪个。

In [ ]:
p   = AiyagariParams()
hh  = aiyagari_household(p)
env = make_env(hh; aiyagari_prices(5.0, p)...)
res = solve_steady_state_given_env!(hh, env)

@printf "K_supplied = %.4f  (K_guess = 5.0)\n" res.moments.K_supplied
@printf "VFI iters: %d   Λ iters: %d\n" res.history.vfi_iters res.history.lambda_iters

### 2.5 Steady state via tatonnement on K / 用 tâtonnement 求稳态 $\bar K$

Outer loop: guess $K$, get $K^{\mathrm{supplied}}$ from the household
block, damped-update $K$, repeat. The chain's buffer warm-starts each
call automatically, so the inner solves get cheaper as the outer
iteration progresses.

外层循环：猜 $K$，从家庭模块拿到 $K^{\mathrm{supplied}}$，
阻尼更新 $K$，反复迭代。链的 buffer 会自动热启动，
内层求解会越来越快。

In [ ]:
function aiyagari_steady_state(p::AiyagariParams;
                               K_init   = 5.0,
                               damp     = 0.01,
                               rtol     = 2e-2,
                               max_iter = 500,
                               verbose  = false)
    hh = aiyagari_household(p)
    K  = K_init
    local V, Λ
    history = Float64[]
    for it in 1:max_iter
        env = make_env(hh; aiyagari_prices(K, p)...)
        res = solve_steady_state_given_env!(hh, env)
        V, Λ = res.V, res.Λ
        K_S  = res.moments.K_supplied
        err  = abs(K_S - K) / K
        push!(history, err)
        verbose && @printf "  iter %3d: K = %.4f  K_S = %.4f  err = %.4e\n" it K K_S err
        err ≤ rtol && return (; K, V, Λ, hh, history, iters = it)
        K += damp * (K_S - K)
    end
    error("aiyagari_steady_state: tatonnement did not converge in $max_iter iterations")
end

ss = aiyagari_steady_state(p; verbose = false)
@printf "K_ss = %.4f, r = %.4f, w = %.4f  (in %d outer iters)\n" ss.K aiyagari_prices(ss.K, p).r aiyagari_prices(ss.K, p).w ss.iters

### 2.6 Comparative statics / 比较静态

Permanent 5% positive TFP shock. We swap `p.A` with `@set` and
re-run the tatonnement starting from the old SS (a good warm start).

永久性 5% 正向 TFP 冲击。用 `@set` 替换 `p.A`，
以旧稳态为起点（很好的热启动）重新跑 tâtonnement。

In [ ]:
ss_old = aiyagari_steady_state(p)
p_new  = @set p.A = 1.05
ss_new = aiyagari_steady_state(p_new; K_init = ss_old.K * 1.05)

@printf "ΔK = %+0.4f  (%.2f%%)\n" (ss_new.K - ss_old.K) 100*(ss_new.K/ss_old.K - 1)
@printf "old K_ss = %.4f   new K_ss = %.4f\n" ss_old.K ss_new.K

## 3. MIT shock — perfect-foresight transition
## 3. MIT 冲击 —— 完全预期下的过渡

**Given.** The exogenous path $\{A_t\}_{t=1}^T$, the household chain
`hh`, and the firm $(\alpha, \delta, L)$.

**Find.** Sequences $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ such that

- $V_{T+1} = V_{\mathrm{ss\_new}}$ (terminal condition),
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$ (initial condition),
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$ for $t = T, T{-}1, \ldots, 1$,
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t, V_{t+1}, \mathrm{env}_t)$ for $t = 1, 2, \ldots, T$,
- $K_t = \int b\,\mathrm{d}\Lambda_{t+1} = K_t^{\mathrm{supplied}}$ (market clears every period).

**Algorithm.** Guess $\{K_t\}$, sweep $V$ backward from the terminal
condition, sweep $\Lambda$ forward from the initial condition, read
off $K_t^{\mathrm{supplied}}$, damped-update $\{K_t\}$.

**已知。** 外生路径 $\{A_t\}_{t=1}^T$、家庭链 `hh`、企业 $(\alpha, \delta, L)$。

**求。** 序列 $\{K_t\}, \{V_t\}, \{\Lambda_t\}$ 满足

- $V_{T+1} = V_{\mathrm{ss\_new}}$（末端条件），
- $\Lambda_1 = \Lambda_{\mathrm{ss\_old}}$（初始条件），
- $V_t = \texttt{backward!}(V_{t+1}, \mathrm{env}_t)$，$t = T, T{-}1, \ldots, 1$，
- $\Lambda_{t+1} = \texttt{forward!}(\Lambda_t, V_{t+1}, \mathrm{env}_t)$，$t = 1, 2, \ldots, T$，
- $K_t = \int b\,\mathrm{d}\Lambda_{t+1} = K_t^{\mathrm{supplied}}$（每期市场出清）。

**算法。** 猜 $\{K_t\}$，从末端条件向后扫 $V$，从初始条件向前扫 $\Lambda$，
读出 $K_t^{\mathrm{supplied}}$，阻尼更新 $\{K_t\}$。

### 3.1 Clean version — using `solve_transition_given_env_path!` / 简洁版本 —— 用 `solve_transition_given_env_path!`

`solve_transition_given_env_path!` owns the per-period buffer allocation, the
backward and forward sweeps with kernel-cache-correct re-seating,
and the per-period moment computation. The outer loop just supplies
the env path and the boundary conditions.

`solve_transition_given_env_path!` 负责按期分配 buffer、做后向和前向扫描
（顺带处理核缓存的正确刷新）、计算每期的矩。外层循环只需要
提供 env 路径和边界条件。

In [ ]:
function mit_shock_transition(p::AiyagariParams;
                              A_new    = 1.05,
                              T        = 100,
                              damp     = 0.2,
                              tol      = 1e-3,
                              max_iter = 200,
                              verbose  = false)
    # 1-2. Endpoint steady states
    # 1-2. 两端的稳态
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init = pre.K * A_new)

    hh      = aiyagari_household(p_new)
    K_path  = collect(range(pre.K, post.K; length = T))
    history = Float64[]

    for it in 1:max_iter
        # 3a. Build the env path from the current K guess
        # 3a. 用当前 K 的猜测构造 env 路径
        env_path = [make_env(hh; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # 3b. One backward + one forward sweep, with the boundary conditions explicit
        # 3b. 一次后向扫 + 一次前向扫，边界条件显式写出
        res = solve_transition_given_env_path!(hh, env_path; Λ_0 = pre.Λ, V_T = post.V)

        # 3c. Residual + damped update
        # 3c. 残差 + 阻尼更新
        K_S = [m.K_supplied for m in res.moments_path]
        err = maximum(abs.(K_S .- K_path))
        push!(history, err)
        verbose && (it ≤ 5 || it % 5 == 0) &&
            @printf "  iter %3d: ‖K^S − K‖∞ = %.4e\n" it err

        err ≤ tol && return (; K_path, K_S, V_path = res.V_path, Λ_path = res.Λ_path,
                              pre, post, history, iters = it)
        K_path .= (1 - damp) .* K_path .+ damp .* K_S
    end
    error("mit_shock_transition: outer tatonnement did not converge in $max_iter iterations")
end

T  = 100
tr = mit_shock_transition(p; A_new = 1.05, T = T, verbose = true)
@printf "\nConverged in %d outer iters.\n" tr.iters
@printf "K_ss_pre  = %.4f\n" tr.pre.K
@printf "K_ss_post = %.4f\n" tr.post.K
@printf "K[1]   (impact) = %.4f\n" tr.K_path[1]
@printf "K[5]            = %.4f\n" tr.K_path[5]
@printf "K[20]           = %.4f\n" tr.K_path[20]
@printf "K[end] (≈post)  = %.4f\n" tr.K_path[end]

### 3.2 Manual version — what's inside `solve_transition_given_env_path!` / 手写版本 —— `solve_transition_given_env_path!` 里头到底是什么

Same algorithm with every step exposed. Useful pedagogically: the
clean version *is* this loop, just packaged.

Three things this version makes visible:

- **Per-period chains** — `hh_path[t]` is one chain per period,
  sharing the Spec but each with its own Buffer. That way each
  period's backward result (the kernel) is preserved for the matching
  forward sweep, without redoing the work. A single chain would also
  work — the kernel cache would re-seat by re-running `backward!`
  whenever `forward!` saw a mismatched `V_end` — but that doubles the
  per-iteration work.
- **Boundary conditions as array endpoints** — `V_path[T+1] = post.V`
  and `Λ_path[1] = pre.Λ` sit at the array endpoints, taken directly
  from the steady-state returns. No accessor into chain internals.
- **`backward!` and `forward!` return their outputs** — `V_path[t]`
  flows out of `backward!`, `Λ_path[t+1]` flows out of `forward!`.
  Per-period moments come from `compute_moments(hh, Λ, env)`, with
  `Λ` passed in explicitly.

同样的算法，把每一步都摊开来给你看。教学上有用：上面那个简洁版本
*就是*这个循环，只是被打了个包。

这个版本把三样东西暴露了出来：

- **按期一条链** —— `hh_path[t]` 每期一条，共用同一个 Spec，
  但各有自己的 Buffer。这样每期 `backward!` 得到的核，就能留到
  对应的 `forward!` 用，不用重算。用同一条链也能跑——核缓存会在
  `forward!` 看到不匹配的 `V_end` 时重跑 `backward!` 自动刷新——
  但每轮要做两倍的工。
- **边界条件就是数组两端** —— `V_path[T+1] = post.V` 和
  `Λ_path[1] = pre.Λ` 直接从稳态结果里塞进数组两头，不需要去
  拨弄链的内部状态。
- **`backward!` 与 `forward!` 把输出返回出来** —— `V_path[t]`
  从 `backward!` 流出来，`Λ_path[t+1]` 从 `forward!` 流出来。
  每期的矩用 `compute_moments(hh, Λ, env)` 算，`Λ` 是显式传进去的。

In [ ]:
function mit_shock_transition_manual(p::AiyagariParams;
                                     A_new    = 1.05,
                                     T        = 100,
                                     damp     = 0.2,
                                     tol      = 1e-3,
                                     max_iter = 200,
                                     verbose  = false)
    # Endpoint steady states
    # 两端的稳态
    pre   = aiyagari_steady_state(p)
    p_new = @set p.A = A_new
    post  = aiyagari_steady_state(p_new; K_init = pre.K * A_new)

    # One chain per period — same spec, fresh buffer.
    # 每期一条链 —— 同一个 spec，各一份 buffer。
    hh_path = [aiyagari_household(p_new) for _ in 1:T]
    dims    = layout_size(aiyagari_layout(p_new))

    # V_path[t]   = continuation value at the start of period t
    # V_path[T+1] = post.V    (terminal boundary)
    # Λ_path[t]   = distribution at the start of period t
    # Λ_path[1]   = pre.Λ     (initial boundary)
    V_path = [zeros(Float64, dims...) for _ in 1:T+1]
    Λ_path = [zeros(Float64, dims...) for _ in 1:T+1]
    copyto!(V_path[T+1], post.V)
    copyto!(Λ_path[1],   pre.Λ)

    K_path  = collect(range(pre.K, post.K; length = T))
    history = Float64[]

    for it in 1:max_iter
        env_path = [make_env(hh_path[t]; aiyagari_prices(K_path[t], p_new)...) for t in 1:T]

        # Backward sweep: V_t = backward!(V_{t+1}, env_t) on chain hh_path[t].
        # backward! returns V_start; copy it into V_path[t].
        # 后向扫描：V_t = backward!(V_{t+1}, env_t)，写到 V_path[t]。
        for t in T:-1:1
            copyto!(V_path[t], backward!(hh_path[t], V_path[t+1], env_path[t]))
        end

        # Forward sweep: Λ_{t+1} = forward!(Λ_t) on chain hh_path[t].
        # The kernel cache is fresh (we just ran backward on this chain), so the
        # cheap forward call is correct by construction.
        # 前向扫描：Λ_{t+1} = forward!(Λ_t)。核缓存刚刚被同一条链的 backward 刷过，
        # 直接用就行。
        K_S = zeros(T)
        for t in 1:T
            copyto!(Λ_path[t+1], forward!(hh_path[t], Λ_path[t]))
            K_S[t] = compute_moments(hh_path[t], Λ_path[t+1], env_path[t]).K_supplied
        end

        err = maximum(abs.(K_S .- K_path))
        push!(history, err)
        verbose && (it ≤ 5 || it % 5 == 0) &&
            @printf "  iter %3d: ‖K^S − K‖∞ = %.4e\n" it err

        err ≤ tol && return (; K_path, K_S, V_path, Λ_path, pre, post, history, iters = it)
        K_path .= (1 - damp) .* K_path .+ damp .* K_S
    end
    error("mit_shock_transition_manual: outer tatonnement did not converge in $max_iter iterations")
end

tr_manual = mit_shock_transition_manual(p; A_new = 1.05, T = T)
@printf "Manual converged in %d outer iters.\n" tr_manual.iters
@printf "‖K_path_clean − K_path_manual‖∞ = %.2e\n" maximum(abs.(tr.K_path .- tr_manual.K_path))

### 3.3 Capital IRF / 资本的脉冲响应

The aggregate-capital path after the +5% TFP shock, with the pre-
and post-shock steady states marked as dashed horizontal lines.

+5% TFP 冲击后总资本的路径；横向虚线是冲击前后两个稳态的 K 水平。

In [ ]:
plot(1:T, tr.K_path; lw = 2, label = "K_t",
     xlabel = "period t", ylabel = "aggregate capital K",
     title  = "IRF: K to a +5% permanent TFP shock")
hline!([tr.pre.K];  color = :gray, linestyle = :dash, label = "K_ss^pre")
hline!([tr.post.K]; color = :gray, linestyle = :dot,  label = "K_ss^post")

### 3.4 Real-rate and wage paths / 利率与工资的路径

Read prices off the K path with `aiyagari_prices`, using the new
(post-shock) TFP parameter.

把 K 路径代入 `aiyagari_prices`（用新的 TFP 参数）就拿到了
利率与工资的路径。

In [ ]:
prices_path = [aiyagari_prices(K, p_new) for K in tr.K_path]
r_path = [pr.r for pr in prices_path]
w_path = [pr.w for pr in prices_path]

plot(layout = (2, 1), size = (700, 500))
plot!(1:T, r_path; subplot = 1, lw = 2, label = "r_t",
      xlabel = "period", ylabel = "r", title = "IRF: real rate")
hline!([aiyagari_prices(tr.pre.K,  p     ).r]; subplot = 1, color = :gray, linestyle = :dash, label = "r_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).r]; subplot = 1, color = :gray, linestyle = :dot,  label = "r_ss^post")

plot!(1:T, w_path; subplot = 2, lw = 2, label = "w_t",
      xlabel = "period", ylabel = "w", title = "IRF: wage")
hline!([aiyagari_prices(tr.pre.K,  p     ).w]; subplot = 2, color = :gray, linestyle = :dash, label = "w_ss^pre")
hline!([aiyagari_prices(tr.post.K, p_new ).w]; subplot = 2, color = :gray, linestyle = :dot,  label = "w_ss^post")

### 3.5 Tatonnement residual history / 外层残差的下降过程

Damped tatonnement drops the residual geometrically until it hits a
*discretization floor* at $\sim 2.5\times 10^{-3}$ on the baseline
calibration. The floor is the hard-`argmax` `ConsumptionSavingsStage`
policy flipping between adjacent grid cells as $K_t$ wobbles.
A smoothed (`LogitChoiceStage`-based) savings policy or a tighter
wealth grid would push the floor down.

阻尼 tâtonnement 让残差几何下降，直到撞上 $\sim 2.5\times 10^{-3}$
的*离散化下限*。下限来自硬 `argmax` 的 `ConsumptionSavingsStage`：
$K_t$ 微动时策略在相邻格点间翻转。把储蓄换成平滑
（基于 `LogitChoiceStage`）或加密财富格点，下限就会下降。

In [ ]:
plot(1:length(tr.history), tr.history;
     yscale = :log10, lw = 2, marker = :circle, markersize = 3,
     xlabel = "outer iteration", ylabel = "‖K^S − K‖∞",
     label  = "residual", title = "Damped tatonnement residual history")

### 3.6 Damping sweep / 阻尼参数扫一遍

Damping is a craft: too high oscillates, too low crawls. Re-run the
transition at $d \in \{0.1, 0.2, 0.4, 0.6\}$ and overlay the residual
histories.

阻尼是个手艺活：太大会振荡，太小爬不动。把过渡在
$d \in \{0.1, 0.2, 0.4, 0.6\}$ 跑几遍，把残差历史叠在一张图上。

**Expected.** $d = 0.6$ oscillates or fails to converge; $d = 0.1$
converges slowly; $d = 0.2$–$0.4$ is roughly the sweet spot.

**预期。** $d = 0.6$ 会振荡或不收敛；$d = 0.1$ 收敛很慢；
$d = 0.2$–$0.4$ 大致是最佳区间。

In [ ]:
plt = plot(yscale = :log10, xlabel = "outer iteration",
           ylabel = "residual ‖K^S − K‖∞",
           title  = "Damping sweep")
for d in (0.1, 0.2, 0.4, 0.6)
    local r
    try
        r = mit_shock_transition(p; A_new = 1.05, T = T, damp = d,
                                  tol = 1e-3, max_iter = 80)
    catch err
        # If a setting fails to converge inside max_iter, the function
        # errors; we still want to plot what residual history it had.
        # 没收敛会抛错；这里把已有的残差历史画出来。
        @warn "d = $d did not converge; plotting partial history."
        continue
    end
    plot!(plt, 1:length(r.history), r.history; lw = 2, label = "d = $d")
end
plt